# Exercise: the distillation loss for one example

**Goal:** compute the top-K self-distillation loss for one example from `runs/distill/examples.jsonl`, by hand, step by step.

- **Teacher:** Qwen3.5-9B with the hint in its context (`teacher_prompt`).
- **Student:** the same model without the hint (`student_prompt`).
- Both read the same **target**: the tokens the model actually wrote at the hint turn.

At every target position $i$, the teacher's top-K tokens, renormalized to sum to 1, give a distribution $p_T(\cdot \mid i)$ that is zero outside those K tokens. The loss pushes the student's **full** distribution $p_S(\cdot \mid i)$, over the whole vocabulary, toward it:

$$\mathcal{L} = \frac{1}{T}\sum_{i=1}^{T} \sum_{k \in \text{top-}K_T(i)} p_T(k \mid i)\,\big(\log p_T(k \mid i) - \log p_S(k \mid i)\big)$$

That's KL(teacher ‖ student), averaged over the $T$ target positions. Since $p_T$ is zero outside its top-K, only those K tokens appear in the sum, but $\log p_S$ is the student's unrenormalized log-prob: probability the student puts on other tokens counts against it. This is the loss training minimizes (step 6 shows how it relates to Tinker's `cross_entropy`).

**Steps**
1. Build the teacher and student sequences, and find where the target sits in each.
2. Get the teacher's top-K tokens and log-probs at the target positions (Tinker).
3. Get the student's log-probs for those same tokens: a Tinker training-client forward pass on a training datum.
4. Compute the loss.
5. Look at where the teacher and student disagree most.
6. Compare with Tinker's own `cross_entropy` loss on the same datum, the one training will use.

Each step has a **TODO** cell for you and a **check** cell after it. Run the check: it tells you whether your code is right.

**Cost:** the default example has an 11.7k-token context and 107 target tokens, so each Tinker call is about a cent.

In [1]:
# Setup (given). Run this first.
import json
import os
from pathlib import Path
import sys

import numpy as np
import tinker
import torch

# Jupyter runs from notebooks/; make the repo root the working directory and importable.
REPO = Path.cwd() if (Path.cwd() / "sd").exists() else Path.cwd().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from sd.renderer import get_agent_renderer


# TINKER_API_KEY from the environment, or from the gitignored .env
if "TINKER_API_KEY" not in os.environ and (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
assert "TINKER_API_KEY" in os.environ, "set TINKER_API_KEY or create .env"

MODEL = "Qwen/Qwen3.5-9B"
K = 20

examples = [json.loads(l) for l in open("runs/distill/examples.jsonl")]
example = next(e for e in examples if e["instance_id"] == "sphinx-doc__sphinx-8638")  # smallest context: cheap

renderer = get_agent_renderer(MODEL)
tok = renderer.tokenizer
service = tinker.ServiceClient()
sampling_client = service.create_sampling_client(base_model=MODEL)       # teacher (step 2)
training_client = service.create_lora_training_client(base_model=MODEL)  # student forward pass (step 3)

print(example["instance_id"], "| hint turn", example["hint_turn"])
print("student prompt", len(example["student_prompt"]), "tokens | teacher prompt", len(example["teacher_prompt"]),
      "| target", len(example["target"]))
print("hint:", example["hint"])
print("target starts:", repr(tok.decode(example["target"][:40])))

/Users/dmorara/repos/applied-compute/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sphinx-doc__sphinx-8638 | hint turn 32
student prompt 11712 tokens | teacher prompt 11754 | target 107
hint: Note: you're close to your 40-turn limit, with about 9 turns left. You often keep exploring and forget to submit. Wrap up and call `task_complete`.
target starts: "Now I see the issue. There's a module-level variable `somevar` and a class attribute `foo.somvar` (instance variable). The automodule documentation might be auto-linking these"


## Step 1: build the sequences

The model reads the prompt followed by the target, so the full sequences are:

- `student_seq = student_prompt + target`
- `teacher_seq = teacher_prompt + target`

The target is the **last `T` tokens of both**, but at different absolute positions, because the teacher's prompt is longer by the hint message (about 42 tokens).

Compute:
- `T`: the number of target tokens
- `student_pos`: the list of positions of the target tokens inside `student_seq`
- `teacher_pos`: the same inside `teacher_seq`

For example, if `student_seq` has length 10 and `T = 3`, then `student_pos == [7, 8, 9]`.

In [3]:
# TODO
target = example["target"]
T = len(target)
student_seq = example['student_prompt'] + target
teacher_seq = example['teacher_prompt'] + target
student_pos = list(range(len(student_seq) - T, len(student_seq)))
teacher_pos = list(range(len(teacher_seq) - T, len(teacher_seq)))

In [4]:
# Check step 1
assert T == len(example["target"])
assert len(student_seq) == len(example["student_prompt"]) + T and len(teacher_seq) == len(example["teacher_prompt"]) + T
assert [student_seq[p] for p in student_pos] == target and [teacher_seq[p] for p in teacher_pos] == target
assert student_pos[-1] == len(student_seq) - 1 and teacher_pos[-1] == len(teacher_seq) - 1
print("step 1 OK")

step 1 OK


## Step 2: the teacher's top-K at the target positions

Ask Tinker for top-K **prompt** log-probs over the whole `teacher_seq`. We don't need any generation, so `max_tokens=1`.

```python
response = sampling_client.sample(
    prompt=tinker.ModelInput.from_ints(teacher_seq),
    num_samples=1,
    sampling_params=tinker.SamplingParams(max_tokens=1),
    include_prompt_logprobs=True,   # required: without it, no top-K comes back
    topk_prompt_logprobs=K,
).result()
response.topk_prompt_logprobs_np.token_ids  # int32,   shape (len(teacher_seq), K)
response.topk_prompt_logprobs_np.logprobs   # float32, shape (len(teacher_seq), K)
```

**Indexing:** row `i` is the teacher's distribution for the token **at position `i`**, given the tokens before it. Row 0 is empty, since nothing comes before it.

Compute, as torch tensors of shape `(T, K)`:
- `teacher_ids`: the teacher's top-K token IDs at each target position
- `teacher_lp`: their log-probs

In [16]:
# TODO
response_t = sampling_client.sample(
    prompt=tinker.ModelInput.from_ints(teacher_seq),
    num_samples=1,
    sampling_params=tinker.SamplingParams(max_tokens=1),
    include_prompt_logprobs=True,
    topk_prompt_logprobs=K
).result()


In [32]:
teacher_ids = response_t.topk_prompt_logprobs_np.token_ids[-T:]
teacher_lp = response_t.topk_prompt_logprobs_np.logprobs[-T:]

In [33]:
# Check step 2
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
teacher_ids, teacher_lp = _t(teacher_ids).long(), _t(teacher_lp).float()
assert teacher_ids.shape == (T, K) and teacher_lp.shape == (T, K)
assert torch.all(teacher_lp <= 0) and torch.all(teacher_lp[:, :-1] >= teacher_lp[:, 1:]), "rows should be sorted, best first"
in_topk = (teacher_ids == torch.tensor(target).unsqueeze(1)).any(1).float().mean().item()
mass = teacher_lp.exp().sum(1)
print(f"step 2 OK | the written token is in the teacher's top-{K} at {in_topk:.0%} of positions | "
      f"top-{K} covers {mass.mean():.1%} of the teacher's probability on average (min {mass.min():.1%})")

step 2 OK | the written token is in the teacher's top-20 at 100% of positions | top-20 covers 99.8% of the teacher's probability on average (min 95.1%)


## Step 3: the student's log-probs for the same tokens

We need the **student's** log-prob for each of the teacher's K tokens at each target position. The sampler can't do this here: it scores at most 20 distinct token IDs per request, and the teacher's top-20 over a whole target involve about 1,000.

The **training client's forward pass** can. It takes a `tinker.Datum`: the input tokens plus, per row, K `target_tokens` to score. It returns the student's log-prob of every one of them:

```python
datum = tinker.Datum(
    model_input=tinker.ModelInput.from_ints(student_seq[:-1]),       # the model predicts token i+1 from tokens <= i
    loss_fn_inputs={
        "target_tokens": tinker.TensorData.from_torch(target_tokens),  # int64, shape (len(student_seq) - 1, K)
        "weights": tinker.TensorData.from_torch(weights),              # float32, same shape
    },
)
out = training_client.forward([datum], loss_fn="cross_entropy").result()   # forward only: no weight update
out.loss_fn_outputs[0]["logprobs"].to_torch()   # shape (len(student_seq) - 1, K): the student's log-prob of each target cell
```

**Indexing:** **row `i` predicts the token at position `i + 1`.** So the target token at position `p` lives in row `p - 1`. (Step 2 was different: there, row `i` was position `i`.)

Fill `target_tokens` with the teacher's top-K IDs on the target rows. Other rows can hold anything, e.g. zeros.

**`weights`:** Tinker's `cross_entropy` requires the key, but its values don't change the log-probs you read here, so use zeros. The weights only define the *loss* (`-weight * logprob` per cell); step 6 fills them in for real.

Compute `student_lp`, shape `(T, K)`: the student's log-probs for `teacher_ids`.

In [57]:
# TODO
target_tokens = torch.concat([torch.zeros(len(student_seq) - T - 1, K, dtype=int), teacher_ids])
weights = torch.concat([torch.zeros(len(student_seq) - T - 1, K), torch.softmax(teacher_lp, 1)])
datum = tinker.Datum(
    model_input=tinker.ModelInput.from_ints(student_seq[:-1]),
    loss_fn_inputs={
        "target_tokens": target_tokens,
        "weights": weights
    }

)
out = training_client.forward([datum], loss_fn="cross_entropy").result()

In [66]:
student_lp = out.loss_fn_outputs[0]['logprobs'].to_torch()[-T:]

In [67]:
# Check step 3
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
student_lp = _t(student_lp).float()
assert student_lp.shape == (T, K) and torch.all(student_lp <= 0)
# The student's log-prob of the token it actually wrote should match what vLLM recorded during the rollout.
match = teacher_ids == torch.tensor(target).unsqueeze(1)              # where the written token sits in each row
has = match.any(1)
written_lp = student_lp[has, match[has].float().argmax(1)]
diff = (written_lp - torch.tensor(example["target_logprobs"])[has]).abs()
print(f"step 3 OK | student vs recorded log-prob of the written token ({has.sum()}/{T} positions): "
      f"median |diff| {diff.median():.4f}, max {diff.max():.3f}")
assert diff.median() < 0.05, "student log-probs don't match the rollout: check the row/position offset"

step 3 OK | student vs recorded log-prob of the written token (107/107 positions): median |diff| 0.0000, max 0.111


## Step 4: the loss

Implement `topk_kl(teacher_lp, student_lp)`. Both inputs have shape `(T, K)` and hold log-probs over the **same** K tokens per row: the teacher's top-K. It returns a tensor of shape `(T,)`: the KL at each position.

1. Renormalize the **teacher** over its K tokens: `log p_T = log_softmax(teacher_lp)` per row, and `p_T = log p_T.exp()`. The top-K covers most but not all of the teacher's probability; renormalizing makes $p_T$ a proper distribution. Use the renormalized $\log p_T$ in the formula too, not the raw `teacher_lp`, or the KL can go negative.
2. Leave the **student** as it is: `student_lp` holds its full-vocabulary log-probs, which is what it samples from. Don't renormalize it over the K tokens.
3. Per row: $\sum_k p_T(k)\,(\log p_T(k) - \log p_S(k))$.

Then `loss = per_position.mean()`, the average over target positions.

**Why not renormalize the student too?** That gives a different KL, over the K tokens only. With $s$ = the student's total probability on the teacher's K tokens:

$$\text{KL}_\text{full} = \text{KL}_\text{shape} + (-\ln s)$$

$\text{KL}_\text{shape}$ only compares how the two rank the teacher's K tokens. It can be 0 while the student puts 90% of its probability elsewhere, and the student samples from its full distribution, so it would rarely write what the teacher wants. The $-\ln s$ term catches that.

In [ ]:
# TODO
def topk_kl(teacher_lp: torch.Tensor, student_lp: torch.Tensor) -> torch.Tensor:
    ...

per_position = topk_kl(teacher_lp, student_lp)
loss = per_position.mean()

In [ ]:
# Check step 4
# Accept numpy arrays too (np.ascontiguousarray handles views like a[:, ::-1]).
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
per_position = _t(per_position).float()
t = torch.log(torch.tensor([[0.5, 0.3, 0.2]])); s = torch.log(torch.tensor([[0.2, 0.3, 0.5]]))
assert torch.allclose(topk_kl(t, s), torch.tensor([0.274887]), atol=1e-5), topk_kl(t, s)          # worked out by hand
assert torch.allclose(topk_kl(t, t), torch.zeros(1), atol=1e-6)                                    # identical -> 0
assert torch.allclose(topk_kl(t + np.log(0.9), s), topk_kl(t, s), atol=1e-5), \
    "the teacher's top-K covering only 90% shouldn't matter: renormalize the teacher (log p_T and p_T)"
assert torch.allclose(topk_kl(t, s + np.log(0.5)), topk_kl(t, s) + np.log(2), atol=1e-5), \
    "the student putting half its probability elsewhere should cost -ln 0.5 = 0.693: don't renormalize the student"
assert torch.allclose(topk_kl(t, t + np.log(0.1)), torch.tensor([2.302585]), atol=1e-5)          # same shape, s = 0.1 -> -ln 0.1
assert per_position.shape == (T,) and torch.all(per_position >= -1e-6)
print(f"step 4 OK | loss {loss.item():.4f} | per position: median {per_position.median():.4f}, max {per_position.max():.3f}")

## Step 5: where do the teacher and student disagree?

Using `per_position`, find the **10 positions with the highest KL**. For each, print one line with:
- the position within the target, and the written token (decode it with `tok.decode([id])`)
- the KL
- the teacher's top 3 tokens with their probabilities (renormalized), next to the student's probabilities for those same tokens
- `s`: the student's total probability on the teacher's top-K

For example:

```
pos   0 | KL 0.853 | wrote 'Now' | teacher: 'Let' 0.78, 'Now' 0.12, 'I' 0.07 | student: 'Let' 0.xx, 'Now' 0.xx, 'I' 0.xx | s 0.xx
```

Questions to answer:
- Are the big disagreements at the start of the turn, for example its first thinking tokens, or spread out?
- Does the teacher's preferred alternative look like wrapping up (*the fix works*, *review the diff*, *submit*) while the written token keeps exploring (*also*, *test*, *check*)?
- Is a big disagreement about **ranking** (the student prefers a different token among the teacher's top-K) or about **missing mass** (low `s`: the student prefers tokens the teacher doesn't consider)?
- *Bonus:* plot `per_position` against position within the target.

In [111]:
# TODO
values, indices = per_position.topk(10)

for i in range(10):
    idx = indices[i]
    target_id = target[idx]
    print(target_id)
    target_tok = tok.decode(target_id)
    print(target_tok)
    print(teacher_lp[idx, :3])
    print(tok.decode(teacher_ids[idx, :3]))
    print()


6820
Now
tensor([-0.2462, -2.1212, -2.6212])
LetNowI

1436
 see
tensor([-0.3706, -1.7456, -2.4956])
 understand need have

2532
 There
tensor([-0.6207, -1.6207, -1.8707])
 The In There

4845
 autom
tensor([-1.1274, -1.7524, -1.8774])
 issue autod problem

1380
 try
tensor([-0.6468, -1.8968, -2.3968])
 build try look

353
 I
tensor([-0.5276, -0.9026, -5.5276])
 let I,

198


tensor([-0.2247, -1.7247, -4.0997])



 But

13
.
tensor([-0.5311, -1.7811, -2.4061])
.! clearly

279
 the
tensor([-0.0287, -5.0287, -5.4037])
 the -.

4125
 issue
tensor([-0.1733, -3.0483, -3.2983])
 issue problem example



## Step 6: the training datum and Tinker's loss

Training will call `training_client.forward_backward([datum], loss_fn="cross_entropy")`, which computes, per cell, `-weight * logprob`, and sums. To make that the distillation loss, build the **training datum**: the same as step 3's, but with real weights.

- `train_weights`: the teacher's top-K **probabilities**, renormalized to sum to 1 per row (that's $p_T$ from step 4), on the target rows, and `0` elsewhere. Shape `(len(student_seq) - 1, K)`.
- `train_datum`: step 3's datum with `train_weights` as `"weights"`.
- `train_out = training_client.forward([train_datum], loss_fn="cross_entropy").result()`: a forward pass only, so it doesn't train. Training would call `forward_backward` on exactly this datum.

`train_out` gives you:
- `train_out.loss_fn_outputs[0]["elementwise_loss"]`, shape `(len(student_seq) - 1, K)`: per cell, `-weight * logprob`. Rows with weight 0 are 0.
- `train_out.metrics["loss:sum"]`: the sum of all cells. A **sum** over positions, not a mean.

That's cross-entropy with soft targets: the weights are $p_T$, and the log-probs are the student's full-vocabulary ones. Per position:

$$\text{CE} = -\sum_k p_T(k)\log p_S(k) = \underbrace{-\sum_k p_T(k)\log p_T(k)}_{H(p_T),\ \text{the teacher's entropy}} + \text{KL}_\text{full}$$

The teacher is frozen, so $H(p_T)$ is a constant: it has no gradient, and minimizing CE is minimizing your step-4 KL. It does mean the CE never reaches 0; its floor is $H$.

Compute, as tensors of shape `(T,)` except where noted:
- `ce_per_pos`: CE per target position, from Tinker's `elementwise_loss`
- `ce_sum`: CE summed over target positions, **recomputed by you** from `student_lp` and $p_T$ (a number)
- `H`: the teacher's entropy per position
- `kl_shape` and `missing`: the two parts of your KL, $\text{KL}_\text{shape}$ (student renormalized over the K tokens) and $-\ln s$

Questions to answer:
- Which part dominates the KL on this example: `kl_shape` or `missing`? And at the top-10 positions from step 5?
- How big is $H$ next to the KL? What would the training loss curve look like if the student matched the teacher perfectly?
- `loss:sum` grows with $T$. In a batch with a 107-token and a 2,000-token target, which example drives the gradient? Is that what we want?

For reference, the cookbook builds the same datum in `_collect_topk_for_datum` in `.venv/lib/python3.12/site-packages/tinker_cookbook/distillation/train_off_policy.py`.

In [ ]:
# TODO

In [ ]:
# Check step 6
_t = lambda x: torch.as_tensor(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
train_weights = _t(train_weights).float()
ce_per_pos, H, kl_shape, missing = (_t(x).float() for x in (ce_per_pos, H, kl_shape, missing))
ce_sum = float(ce_sum)
assert train_weights.shape == (len(student_seq) - 1, K)
assert torch.all(train_weights[:-T] == 0) and torch.allclose(train_weights[-T:].sum(1), torch.ones(T), atol=1e-4), \
    "weights: p_T on the target rows (each summing to 1), 0 elsewhere"
assert all(x.shape == (T,) for x in (ce_per_pos, H, kl_shape, missing))
assert abs(ce_sum - train_out.metrics["loss:sum"]) < 1e-2 * max(1.0, abs(ce_sum)), (ce_sum, train_out.metrics["loss:sum"])
assert torch.allclose(ce_per_pos.sum(), torch.tensor(ce_sum), rtol=1e-3)
assert torch.allclose(ce_per_pos, H + per_position, atol=1e-3), "CE should be H + your step-4 KL at every position"
assert torch.allclose(per_position, kl_shape + missing, atol=1e-3), "KL_full should be KL_shape + (-ln s)"
assert torch.all(kl_shape >= -1e-6) and torch.all(missing >= -1e-6)
print(f"step 6 OK | per position, mean: CE {ce_per_pos.mean():.4f} = H {H.mean():.4f} + KL {per_position.mean():.4f} "
      f"(KL_shape {kl_shape.mean():.4f} + missing {missing.mean():.4f})")